# Biohub cell tracking — inference

Predicts lineage graphs for every test movie and writes `submission.csv`.
Runs on Kaggle (2×T4, internet off) and locally; paths come from `SETTINGS.json`.

`VARIANT` selects the model set: `"fin"` (the selected final submission, default) or `"best"`.

## 1. Install

On Kaggle the code dataset holds the `biohub_tracking` wheel and its pinned dependency
wheels; both are installed offline. Locally the package is imported from `src/`
(or installed with `pip install -e .`).

In [ ]:
import os
# Set before torch initialises CUDA: avoids fragmentation OOMs on 16 GB GPUs.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import subprocess
import sys
import zipfile
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
VARIANT = "fin"

if IS_KAGGLE:
    candidates = [Path("/kaggle/input/datasets/jamalsaeedi/biohub-cell-tracking-kaggle"),
                  Path("/kaggle/input/biohub-cell-tracking-kaggle")]
    CODE_DIR = next((c for c in candidates if c.exists()), candidates[0])
    src = CODE_DIR / "src"
    if not src.is_dir():  # a directory upload can arrive as <name>.zip
        src = Path("/kaggle/working/biohub_artifacts/src")
        if not src.exists():
            src.mkdir(parents=True)
            with zipfile.ZipFile(CODE_DIR / "src.zip") as bundle:
                bundle.extractall(src)
    wheel = sorted(src.glob("biohub_tracking-*.whl"))[-1]
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheel)],
                   check=True)
    SETTINGS_PATH = CODE_DIR / "SETTINGS.json"
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    sys.path.insert(0, str(REPO_ROOT / "src"))
    CODE_DIR = REPO_ROOT
    SETTINGS_PATH = None  # BIOHUB_SETTINGS, else the nearest SETTINGS.json upward
print(f"IS_KAGGLE={IS_KAGGLE}  CODE_DIR={CODE_DIR}")

In [ ]:
from biohub_tracking.isotropic.kaggle import bootstrap, versions

environment = bootstrap(CODE_DIR)
print(environment.describe())
print(versions())

## 2. Configuration

`shipped_config` returns the submitted recipe with the variant's six checkpoints and six
member-own edge re-scorers attached, after checking every file's sha256.

In [ ]:
from biohub_tracking.recipe import VARIANTS, shipped_config
from biohub_tracking.settings import load_settings

settings = load_settings(SETTINGS_PATH)
print(settings.describe())
config = shipped_config(VARIANT, list(settings.model_dirs))
spec = VARIANTS[VARIANT]
d, s = config.detection, config.solver
print(f"variant      {VARIANT}  ({spec.kaggle_handle}, sha256 verified)")
for run, views, rescorer in zip(spec.runs, d.member_tta_views, spec.rescorers):
    print(f"  {run:32s} {views} TTA views   re-scorer {rescorer}")
print(f"detection    threshold {d.threshold}")
print(f"solver       disappearance {s.disappearance_weight}, appearance bias {s.appearance_bias}, "
      f"association temperature {s.association_temperature}")
print(f"smoothing    line fit, window {config.smoothing.window}, weight {config.smoothing.weight}, "
      f"drift-compensated {config.smooth_drift_compensated}")

## 3. Run

One worker process per visible GPU; each movie's decode adapts to the remaining session
time, and a movie that fails is re-run on simpler settings, so every movie gets a
prediction. `run_pipeline` writes `submission.csv` and validates its structure.

In [ ]:
from biohub_tracking.cli import run, summarize

submission_path, timing = run(VARIANT, settings)
print(f"\nSubmission written and validated at: {submission_path}")
print(f"  {timing['nodes']} nodes, {timing['edges']} edges, {timing['divisions']} divisions")
print(f"  predict {timing['predict_seconds'] / 60:.1f} min, total {timing['total_seconds'] / 60:.1f} min")

## 4. Check

Nodes, edges and divisions per movie.

In [ ]:
print(summarize(submission_path))